# 딥러닝·시계열 예상문제 200 + 파라미터 계산 20

- **찾는 법:** Jupyter에서 `Ctrl+F`로 키워드를 검색하세요 (예: `LSTM`, `패딩`, `ARIMA`, `편향`).
- **구성:** 0장 기본 개념 → 1~9장 예상문제 200 → 10장 파라미터·출력 크기 계산 20문제 (풀이와 확인 코드 포함)
- ★ 표시는 특히 자주 나올 만한 문제예요.


## 0. 먼저 알아야 할 기본 개념

문제를 풀기 전에 이 페이지만 먼저 익히면 200문제 대부분을 더 쉽게 풀 수 있어요. 괄호 안의 비유는 이해를 돕기 위한 것이고, 시험 답안에는 정의를 쓰세요.


### 딥러닝 한 줄 요약

입력에 가중치를 곱하고 편향을 더해(z = Wx + b) 활성화 함수를 통과시키는 일을 여러 층 반복해 예측값을 만들고, 예측과 정답의 차이(손실)가 줄어드는 방향으로 W와 b를 조금씩 고치는 것.


### 꼭 알아야 할 용어

| 용어 | 뜻 | 비유 |
| --- | --- | --- |
| 가중치(W) | 입력이 결과에 얼마나 중요한지 나타내는 값 | 재료별 넣는 양 |
| 편향(b) | 결과를 전체적으로 옮기는 값 | 기본 간 |
| 활성화 함수 | 층의 출력을 비선형으로 바꾸는 함수 | 다음 층에 얼마나 전달할지 정하는 문지기 |
| 손실 함수 | 예측과 정답의 차이를 숫자로 나타낸 것 | 시험에서 틀린 점수 |
| 순전파 | 입력 → 출력 방향으로 예측값을 계산 | 문제를 푸는 것 |
| 역전파 | 출력 → 입력 방향으로 체인룰을 써서 각 가중치의 기울기를 계산 | 채점 후 어느 부분을 틀렸는지 거슬러 찾기 |
| 경사하강법 | W = W − 학습률 × 기울기 | 안개 냀 산에서 발끝으로 경사를 느끼며 내려가기 |
| 학습률 | 한 번에 W를 얼마나 바꿀지 | 보폭 (너무 크면 골짜기를 넘어감, 너무 작으면 느림) |
| 에포크 | 전체 훈련 데이터를 한 번 다 학습한 것 | 문제집 한 번 다 풀기 |
| 배치 크기 | 가중치를 한 번 고칠 때 쓰는 데이터 수 | 몇 문제 풀고 채점할지 |
| step(iteration) | 한 에포크 안의 업데이트 횟수 = 데이터 수 ÷ 배치 크기 | 60,000 ÷ 32 = 1,875 |
| 과적합 | 훈련 데이터는 잘 맞히지만 새 데이터는 못 맞히는 상태 | 기출문제 답만 외운 학생 |
| 하이퍼파라미터 | 사람이 정해 주는 값(층 수, 노드 수, 학습률, 배치 크기 등) | 학습으로 바뀌는 것은 W와 b뿐 |


### 데이터 나누기와 전처리

- 훈련 데이터는 공부용, 검증 데이터는 모의고사(과적합 확인·조기 종료), 테스트 데이터는 수능(마지막에 한 번만).
- 스케일링: 값의 범위를 맞춘다. 이미지는 255로 나눠 0\~1로 만든다.
- 원핫인코딩: 클래스 번호 2 → [0,0,1,0,...]. 원핫이면 categorical, 정수 그대로면 sparse_categorical.


### 문제 종류별 출력층과 손실 함수

| 문제 | 출력 노드 수 | 출력층 활성화 | 손실 함수 |
| --- | --- | --- | --- |
| 회귀 (집값) | 1 | 없음(linear) | mse |
| 이진분류 (긍정/부정) | 1 | sigmoid | binary_crossentropy |
| 다중분류 (숫자 0\~9) | 클래스 수 | softmax | categorical_crossentropy (정수 레이블이면 sparse_) |


### 활성화 함수 비교

| 함수 | 출력 범위 | 특징 | 주로 쓰는 곳 |
| --- | --- | --- | --- |
| 계단 | 0 또는 1 | 기울기가 0이라 학습 불가 | 퍼셉트론 설명용 |
| sigmoid | 0\~1 | 깊으면 기울기 소실 | 이진분류 출력층, LSTM 게이트 |
| tanh | −1\~1 | 0 중심, 시그모이드보다 소실 적음 | RNN 은닉 상태 |
| ReLU | 0\~∞ | 빠르고 소실 적음, 음수면 죽은 ReLU | 은닉층(기본) |
| Leaky ReLU | −∞\~∞ | 음수에서도 작은 기울기 | 죽은 ReLU 대책 |
| softmax | 0\~1, 합 1 | 확률로 해석 | 다중분류 출력층 |


### 옵티마이저 한 줄 정리

- 배치 GD(전체로 1번) → SGD(1개씩, 빠르지만 흔들림) → 미니배치(중간, 가장 많이 씀)
- 모멘텀 = 관성 (방향), Adagrad·RMSprop = 변수마다 학습률 조절, Adam = 모멘텀 + RMSprop


### 과적합 막기

데이터 늘리기, 모델 단순하게, 가중치 규제(Ridge·Lasso), 드롭아웃(학습 때만 뉴런을 랜덤으로 끔), 조기 종료(검증 손실이 더 안 줄면 멈춤).


### 외워 둘 계산 공식

| 무엇 | 공식 | 예 |
| --- | --- | --- |
| Dense 파라미터 | 입력 수 × 출력 수 + 출력 수 | 784→128: 100,480 |
| Conv2D 파라미터 | 필터 수 × (커널 높이 × 커널 너비 × 입력 채널) + 필터 수 | Conv2D(16,3), 흑백: 160 |
| 특성 맵 크기 | (입력 − 커널 + 2×패딩) ÷ 스트라이드 + 1 | 28, 3×3 → 26 |
| 풀링 후 크기 | 입력 ÷ 2 (2×2 풀링, 소수점 버림) | 13 → 6 |
| SimpleRNN 파라미터 | 유닛 × (입력 차원 + 유닛 + 1) | 3 × (10+3+1) = 42 |
| LSTM 파라미터 | SimpleRNN 공식 × 4 | 4 × 128 × 139 = 71,168 |
| Embedding 파라미터 | 단어 수 × 벡터 차원 | 10,000 × 16 = 160,000 |

활성화·드롭아웃·풀링·Flatten 층은 파라미터가 0개예요.


### 모델별 핵심 한 줄

- **퍼셉트론 / MLP / DNN**: 단층은 직선 하나로만 나눌 수 있어 XOR을 못 푼다 → 은닉층을 넣으면(MLP) 풀린다 → 은닉층이 2개 이상이면 DNN.
- **CNN**: 이미지를 펼치지 않고 작은 필터로 훑어 특징을 뽑는다(돋보기로 한 칸씩 살피기). 같은 필터를 공유해 가중치가 적고 공간 구조를 지킨다. Conv(특징 추출) → Pooling(크기 줄이기) → Flatten → Dense(분류).
- **RNN**: 이전 시점의 은닉 상태를 다음 시점에 다시 넣어 순서를 기억한다(앞 문장을 기억하며 읽기). 길어지면 앞 내용을 잊는다(장기 의존성).
- **LSTM**: 셀 상태(장기 기억) + 세 게이트(입력·삭제·출력)로 무엇을 기억하고 잊을지 정해 장기 의존성을 줄인다(메모장에 쓸지 지울지 고르기).
- **양방향 RNN**: 앞에서부터, 뒤에서부터 두 번 읽어 둘 다 쓴다(빈칸 채우기는 뒷문장도 봐야 쉽다).


### 시계열 핵심

- **정상성**: 평균·분산이 시간에 따라 변하지 않는 상태. 추세나 계절성이 있으면 비정상.
- **비정상 → 정상 만들기**: 로그 변환(분산 안정), 차분(추세 제거), 계절성 차분(계절성 제거), 평활법(노이즈 줄이기).
- **분해**: 추세 + 계절성 + 불규칙 요소.
- **결측치**: LOCF(직전 값), NOCB(직후 값), 이동평균, 보간법(선형·다항식·스플라인).
- **Prophet**: 추세 g(t) + 계절성 s(t) + 휴일 h(t) + 오차. 컬럼은 ds와 y.
- **ARIMA(p,d,q)**: AR(과거 값 p개) + I(차분 d번) + MA(과거 오차 q개).
- **나눌 때**: 섞지 않고 시간 순서대로 앞은 학습, 뒤는 테스트.


### TensorFlow / Keras / PyTorch 흐름

- **TF 1.x**: 그래프를 먼저 정의 → Session에서 실행(Lazy). **TF 2.x**: 바로 실행(Eager) + Keras 기본.
- **Keras**: Sequential → add → compile(optimizer, loss, metrics) → fit → evaluate / predict.
- **PyTorch**: zero_grad → 예측·loss → backward → step. 평가는 eval() + no_grad(). GPU는 모델과 데이터 둘 다 .to(device).


## 1. TensorFlow 1.x / 2.x / Keras 기본 (Q1\~20)


**Q1.** ★ TensorFlow에서 'Tensor'와 'Flow'는 각각 무엇을 뜻하나?

- 정답: Tensor는 스칼라·벡터·행렬 같은 모든 데이터, Flow는 그 텐서가 그래프의 엣지를 따라 노드에서 노드로 흐르는 것.


**Q2.** TensorFlow 그래프를 구성하는 두 요소는? 각 역할은?

- 정답: 노드(Node, 값 저장·연산)와 엣지(Edge, 텐서가 전달되는 선).


**Q3.** ★ TF 1.x 코드는 어떤 두 단계로 실행되나?

- 정답: ① 상수·변수·연산 노드와 엣지를 정의(그래프 생성) → ② Session을 만들고 sess.run()으로 실제 연산.


**Q4.** TF 1.x에서 `c = a + b`를 실행한 뒤 `print(c)`를 하면 3.0이 나오나?

- 정답: 아니다. 연산을 '정의'만 한 것이라 Tensor 객체 정보만 출력된다. 값은 `sess.run(c)`를 해야 나온다.


**Q5.** TF 2.x 환경에서 1.x 방식(Session)을 쓰려면 어떤 코드가 필요한가?

- 정답: `tf.compat.v1.disable_eager_execution()` 후 `tf.compat.v1.Session()` 사용.


**Q6.** a=2, b=3, c=4일 때 `a+b*c`와 `(a+b)*c`를 sess.run한 결과는?

- 정답: 14, 20.


**Q7.** Session을 다 쓴 뒤 호출해야 하는 메서드는?

- 정답: `sess.close()` (자원 반납).


**Q8.** ★ TF 2.0이 정식 배포된 때와, 1.x 대비 가장 큰 변화 두 가지는?

- 정답: 2019년 9월 30일. ① Eager Execution(즉시 실행)이 기본, ② Keras를 공식 High-Level API로 지원.


**Q9.** ★ Lazy Evaluation과 Eager Execution의 차이는?

- 정답: Lazy는 그래프를 먼저 만들고 Session에서 나중에 계산(1.x). Eager는 작성한 순서대로 바로 계산(2.x).


**Q10.** TF 2.x에서 삭제된 1.x의 노드는?

- 정답: `tf.placeholder()`.


**Q11.** TF 2.x에서 텐서를 넘파이 배열로, 넘파이 배열을 텐서로 바꾸는 방법은?

- 정답: `텐서.numpy()`, `tf.convert_to_tensor(배열)`.


**Q12.** `tf.constant`와 `tf.Variable`의 차이는?

- 정답: constant는 값이 고정된 상수 노드. Variable은 학습 중 바뀌는 값(가중치·바이어스)을 담는 변수 노드.


**Q13.** 기존(넘파이로 직접 구현한) 딥러닝 아키텍처와 비교해, 텐서플로우 딥러닝 아키텍처는 은닉층과 출력층에 주로 어떤 활성화 함수를 쓰나? 왜 그런가?

- 정답: 기존에는 둘 다 시그모이드였다. 텐서플로우에서는 은닉층에 ReLU(학습 속도가 빠름), 출력층에 소프트맥스(출력을 확률분포로 나타냄)를 주로 쓴다.


**Q14.** ★ 케라스 프로그래밍의 일반적인 순서를 API 이름으로 쓰면?

- 정답: `Sequential()` → `model.add()` → `model.compile()` → `model.fit()` → `evaluate()` / `predict()` → `save()` / `load_model()`.


**Q15.** 케라스에서 모델의 기본 단위는?

- 정답: 층(layer). 레고 블록처럼 쌓아 ANN·CNN·RNN을 만든다.


**Q16.** 프랑수아 숄레가 권장한 것은 native Keras와 tf.keras 중 어느 쪽인가?

- 정답: `tensorflow.keras`(tf.keras).


**Q17.** `model.compile()`에서 지정하는 세 가지 주요 인자는?

- 정답: optimizer, loss, metrics.


**Q18.** `model.fit()`이 돌려주는 hist의 `hist.history`에는 무엇이 들어 있나?

- 정답: 에포크별 loss(와 metrics, 검증 데이터가 있으면 val_loss 등)를 담은 딕셔너리.


**Q19.** `model.evaluate()`의 반환값 res에서 res[0], res[1]은 각각 무엇인가? (metrics=["accuracy"]일 때)

- 정답: res[0]=손실값(loss), res[1]=정확도. 튜플이 아니라 리스트로 나온다.


**Q20.** 저장한 모델 "mnist_keras.keras"를 다시 불러오는 코드는?

- 정답: `tf.keras.models.load_model("mnist_keras.keras")`. 불러온 모델의 evaluate 결과는 저장 전과 같다.


## 2. 선형회귀·로지스틱회귀·수치미분 (Q21\~35)


**Q21.** ★ x=[1..6], y=[3..8]을 Dense(1, activation="linear")로 학습하면 W와 b는 대략 얼마로 수렴하나?

- 정답: W≈1, b≈2 (y = x + 2).


**Q22.** 위 모델의 summary에서 Dense 층의 파라미터 수는?

- 정답: 2개 (가중치 1 + 바이어스 1).


**Q23.** 선형회귀의 손실 함수 `np.sum((t - y)**2) / len(x)`는 무엇인가?

- 정답: 평균제곱오차(MSE).


**Q24.** ★ 수치미분 함수가 쓰는 공식은? (delta_x = 1e-4)

- 정답: 중앙차분 [f(x+Δx) − f(x−Δx)] / (2Δx).


**Q25.** 수치미분 코드에서 `np.nditer(..., flags=["multi_index"], op_flags=["readwrite"])`를 쓰는 이유는?

- 정답: 행렬의 모든 원소를 인덱스와 함께 하나씩 돌면서, 그 자리의 값을 잠깐 바꿔 편미분하기 위해서.


**Q26.** 수치미분 루프 끝의 `x[idx] = tmp_val`은 왜 필요한가?

- 정답: Δx만큼 바꿨던 값을 원래대로 되돌려야 다음 원소의 미분이 정확해진다.


**Q27.** 경사하강법의 업데이트 식은?

- 정답: W = W − α × ∂E/∂W (α = 학습률).


**Q28.** 학습률이 너무 크거나 너무 작으면 각각 어떤 문제가 생기나?

- 정답: 크면 최소점을 넘어 발산, 작으면 학습이 너무 느리거나 끝까지 못 감.


**Q29.** 다변수 회귀(data_01.csv)에서 `loaded_data[:, [-1]]`과 `loaded_data[:, -1]`의 차이는?

- 정답: `[[-1]]`는 2차원 (N,1)을 유지, `[-1]`은 1차원 (N,)이 된다.


**Q30.** 입력이 3개(x1,x2,x3)인 선형회귀에서 W와 b의 shape는?

- 정답: W (3,1), b (1,).


**Q31.** 로지스틱 회귀의 출력 함수와 그 출력 범위는?

- 정답: 시그모이드 1/(1+e^−z), 0\~1.


**Q32.** ★ 로지스틱 회귀의 손실 함수(크로스 엔트로피)를 쓰면?

- 정답: E = −Σ [t·log(y) + (1−t)·log(1−y)].


**Q33.** 크로스 엔트로피 코드에 `delta = 1e-7`을 더하는 이유는?

- 정답: y가 0이 되면 log(0) = −∞가 되므로 이를 막기 위해.


**Q34.** 로지스틱 회귀 predict에서 최종 분류 기준은?

- 정답: 시그모이드 값이 0.5 이상이면 1, 아니면 0.


**Q35.** 공부시간 x=[2..20], 합격 t=[0×6, 1×4] 데이터로 학습한 뒤 x=3, x=17을 넣으면 결과는?

- 정답: x=3 → 0(불합격), x=17 → 1(합격). 경계는 12와 14 사이.


## 3. 퍼셉트론·논리게이트·XOR·DNN 구조 (Q36\~60)


**Q36.** 인공신경망(ANN) 개념을 1943년에 처음 제안한 사람은?

- 정답: 워렌 맥커럭(신경생리학자)과 월터 피츠(수학자).


**Q37.** 퍼셉트론을 1957년에 제안한 사람은?

- 정답: 프랑크 로젠블라트.


**Q38.** 퍼셉트론에서 가중치가 크다는 것은 무엇을 뜻하나?

- 정답: 그 입력값이 결과에 더 중요하다는 뜻.


**Q39.** 퍼셉트론의 임계치를 보통 어떤 기호로 표시하나? 임계치를 넘으면 1, 아니면 0을 내는 함수는?

- 정답: 세타(Θ). 계단 함수(Step function).


**Q40.** ★ 단층 퍼셉트론으로 구현할 수 있는 게이트와 없는 게이트는?

- 정답: 가능 AND·NAND·OR, 불가능 XOR.


**Q41.** ★ 단층 퍼셉트론으로 XOR을 못 만드는 이유는?

- 정답: 단층 퍼셉트론은 직선 하나로 두 영역을 나눌 수 있는(선형 분리) 문제만 풀 수 있는데, XOR은 직선 하나로 나눌 수 없다.


**Q42.** w1=0.5, w2=0.5, b=−0.7일 때 입력 (1,0)과 (1,1)의 출력은? 어떤 게이트인가?

- 정답: (1,0) → −0.2 → 0, (1,1) → 0.3 → 1. AND 게이트.


**Q43.** NAND 게이트의 가중치·편향 예시와 진리표는?

- 정답: w1=−0.5, w2=−0.5, b=0.7. (0,0)(0,1)(1,0) → 1, (1,1) → 0.


**Q44.** OR 게이트 예시 값 [0.6, 0.6, −0.5]에서 (0,0)의 출력은?

- 정답: −0.5 ≤ 0 이므로 0.


**Q45.** ★ XOR을 다른 게이트 조합으로 만드는 방법은?

- 정답: s1 = NAND(x1,x2), s2 = OR(x1,x2)를 구한 뒤 AND(s1, s2).


**Q46.** 입력 4개에 대해 NAND 출력과 OR 출력을 묶으면 새 입력은?

- 정답: [1,0], [1,1], [1,1], [0,1]. 여기에 AND를 하면 0,1,1,0 = XOR.


**Q47.** 단층 LogicGate 클래스로 XOR을 학습하면 loss와 정확도는 어떻게 되나?

- 정답: loss가 약 2.77에서 더 내려가지 않고, 정확도는 0.25\~0.5 수준에 머문다.


**Q48.** ★ loss가 2.7726 근처에서 멈추는 이유는? (힌트: 4 × ln2)

- 정답: 모델이 모든 입력에 확률 0.5를 내버리기 때문. 샘플 4개 × (−ln 0.5) = 4 × 0.693 ≈ 2.77.


**Q49.** 단층 퍼셉트론과 다층 퍼셉트론(MLP)의 차이는?

- 정답: MLP는 입력층과 출력층 사이에 은닉층이 있다.


**Q50.** ★ 심층 신경망(DNN)의 정의는?

- 정답: 은닉층이 2개 이상인 신경망.


**Q51.** ★ 입력 2 → 은닉 6 → 출력 1인 XOR 모델의 W2, b2, W3, b3 shape와 전체 파라미터 수는?

- 정답: W2 (2,6), b2 (6,), W3 (6,1), b3 (1,). 12+6+6+1 = 25개.


**Q52.** 입력 2 → 은닉1 4 → 은닉2 2 → 출력 1 모델의 파라미터 수는?

- 정답: (8+4) + (8+2) + (2+1) = 25개.


**Q53.** 은닉층 노드 수에 정해진 규칙이 있나? 너무 많으면?

- 정답: 정해진 규칙은 없다(하이퍼파라미터). 층과 노드가 많아지면 학습이 느려진다.


**Q54.** 딥러닝 모델의 세 층 이름과, 각 층의 노드 수를 정하는 기준은?

- 정답: 입력층(피처 수), 은닉층(하이퍼파라미터), 출력층(분류는 클래스 수, 회귀는 1).


**Q55.** 피드 포워드 신경망(FFNN)이란?

- 정답: 입력층에서 출력층 방향으로만 연산이 진행되는 신경망. (반대 예: RNN)


**Q56.** 전결합층(FC)이란? 케라스에서는 어떤 클래스인가?

- 정답: 한 층의 모든 뉴런이 이전 층의 모든 뉴런과 연결된 층(밀집층). `Dense()`.


**Q57.** 순전파(Forward Propagation)란?

- 정답: 입력층에서 출력층 방향으로 가중치와 연산하며 예측값을 구하는 과정.


**Q58.** ★ 수치미분 방식의 치명적인 단점과, 그것을 해결한 알고리즘은?

- 정답: 가중치 하나마다 손실을 두 번씩 계산해 시간이 매우 오래 걸린다. 오차역전파(Back Propagation)가 해결.


**Q59.** ★ 오차역전파가 편미분을 나눠 계산하기 위해 쓰는 수학 법칙은?

- 정답: 체인룰(연쇄법칙). ∂E/∂W를 국소 미분들의 곱으로 분해한다.


**Q60.** 활성화 함수가 반드시 비선형이어야 하는 이유는?

- 정답: 선형 함수를 여러 층 쌓아도 결국 선형 함수 하나와 같아서, 층을 쌓는 의미가 없어지기 때문.


## 4. 활성화 함수·손실 함수·옵티마이저·과적합 (Q61\~85)


**Q61.** 계단 함수가 학습에 쓰이지 않는 이유는?

- 정답: 기울기가 항상 0이고 x=0에서 미분이 불가능해 경사하강법이 작동하지 않는다.


**Q62.** ★ 기울기 소실(Vanishing Gradient)이란?

- 정답: 시그모이드처럼 기울기가 0에 가까운 함수를 여러 층 쓰면, 역전파 때 작은 값이 계속 곱해져 앞쪽 층에 기울기가 거의 전달되지 않고 W가 업데이트되지 않는 현상.


**Q63.** 결론적으로 시그모이드를 어디에서 쓰는 것을 지양해야 하나?

- 정답: 은닉층. (이진분류 출력층에서는 쓴다)


**Q64.** tanh의 출력 범위와, 시그모이드보다 나은 점은?

- 정답: −1\~1. 0을 중심으로 해서 변화폭이 더 커 기울기 소실이 덜하다.


**Q65.** ★ ReLU의 수식과 장점 두 가지는?

- 정답: f(x) = max(0, x). ① 양수에서 포화되지 않아 깊은 망에서 잘 작동, ② 단순 임계값이라 연산이 빠름.


**Q66.** ★ 죽은 ReLU(dying ReLU)란? 해결책은?

- 정답: 입력이 음수면 기울기가 0이 되어 뉴런이 다시 살아나지 못하는 문제. Leaky ReLU.


**Q67.** Leaky ReLU의 수식과 a의 일반적인 값은?

- 정답: f(x) = max(ax, x), a는 보통 0.01.


**Q68.** ★ 출력층 활성화 함수: 회귀 / 이진분류 / 다중분류에 각각 무엇을 쓰나?

- 정답: linear(없음) / sigmoid / softmax.


**Q69.** ★ 소프트맥스 출력의 세 가지 특징은?

- 정답: ① 각 값이 0\~1, ② 전부 더하면 1(확률로 해석), ③ 입력값의 크기 순서가 출력 순서와 같다.


**Q70.** 입력 [1, 2, 3]을 소프트맥스에 넣으면 가장 큰 값은 몇 번째 위치인가?

- 정답: 세 번째(약 0.665). 순서가 유지된다.


**Q71.** ★ 손실 함수 고르기: 회귀 / 이진분류 / 다중분류(원핫) / 다중분류(정수 레이블)

- 정답: mse / binary_crossentropy / categorical_crossentropy / sparse_categorical_crossentropy.


**Q72.** ★ sparse_categorical_crossentropy를 쓰는 조건은?

- 정답: 정답이 원핫이 아니라 정수(0, 1, 2 ...)로 되어 있을 때.


**Q73.** 배치 경사 하강법의 장점과 단점은?

- 정답: 전체 데이터로 1에포크에 한 번 업데이트. 글로벌 미니멈을 찾기 쉽지만 시간과 메모리를 많이 쓴다.


**Q74.** SGD는 부드럽게 수렴하나? fit()에서 어떻게 표현하나?

- 정답: 아니다, 랜덤 데이터 1개씩 계산해 변경폭이 불안정하지만 빠르다. `batch_size=1`.


**Q75.** 미니 배치 경사 하강법의 장점은?

- 정답: 전체를 계산하는 것보다 빠르고 SGD보다 안정적이다. 예: `batch_size=32`.


**Q76.** ★ MNIST 60,000개를 batch_size=32로 학습하면 1에포크에 몇 step인가? BMI 15,000개를 batch_size=100으로 하면?

- 정답: 60000/32 = 1875 step. 15000/100 = 150 step.


**Q77.** ★ 모멘텀의 원리와 장점은?

- 정답: 이전 step의 기울기를 일정 비율 반영해 관성을 준다. 로컬 미니멈에서 빠져나올 수 있다. 모멘텀 계수는 보통 0.9.


**Q78.** 아다그라드(Adagrad)의 원리와 단점은?

- 정답: 많이 변한 변수는 학습률을 작게, 덜 변한 변수는 크게. 학습이 오래 진행되면 학습률이 지나치게 떨어진다.


**Q79.** RMSprop은 아다그라드의 단점을 어떻게 개선했나? rho의 역할은?

- 정답: 지수 이동평균으로 최근 기울기를 더 반영. rho는 과거 기울기의 영향을 줄이는 하이퍼파라미터.


**Q80.** ★ Adam은 어떤 두 방법을 합친 것인가?

- 정답: RMSprop(학습률) + 모멘텀(방향).


**Q81.** Adagrad·RMSprop의 epsilon은 무엇을 막기 위한 값인가?

- 정답: 0으로 나누는 것.


**Q82.** ★ 과적합 방지 기법 네 가지는?

- 정답: 데이터 양 늘리기, 모델 복잡도 줄이기, 가중치 규제(Ridge·Lasso), 드롭아웃.


**Q83.** ★ 드롭아웃 비율 0.5의 뜻과, 예측할 때도 쓰나?

- 정답: 학습 때마다 랜덤으로 절반의 뉴런을 끄고 학습. 예측 때는 쓰지 않는다.


**Q84.** 드롭아웃이 과적합을 막는 이유 두 가지는?

- 정답: ① 특정 뉴런(조합)에 과하게 의존하는 것을 막고, ② 매번 다른 신경망을 앙상블하는 효과가 있다.


**Q85.** 테스트 정확도가 훈련 정확도보다 낮게 나왔다(예: Fashion MNIST 훈련 0.89, 테스트 0.87). 어떻게 해석하나?

- 정답: 약간의 과적합. 모델이 훈련 데이터에 조금 더 맞춰져 있다. 차이가 커지면 위 기법을 적용한다.


## 5. Keras 실습: MNIST·BMI·Fashion MNIST·IMDB (Q86\~100)


**Q86.** MNIST의 X_train, y_train shape는? 픽셀 값 범위는?

- 정답: (60000, 28, 28), (60000,). 0\~255 (uint8).


**Q87.** ★ `X_train / 255`는 어떤 전처리인가? 왜 하나?

- 정답: 0\~1로 맞추는 스케일링(민맥스 효과). 값의 크기를 줄여 학습을 안정적으로, 빠르게 한다.


**Q88.** `reshape(60000, 784)`와 `Flatten(input_shape=(28,28))`의 공통점은?

- 정답: 둘 다 28×28 이미지를 784개 1차원 벡터로 펼친다. 앞은 데이터를, 뒤는 모델 안에서 펼친다.


**Q89.** `to_categorical(5, 10)`의 결과는?

- 정답: [0,0,0,0,0,1,0,0,0,0].


**Q90.** 원핫인코딩 방법 세 가지(라이브러리별)는?

- 정답: 사이킷런 OneHotEncoder, 판다스 get_dummies, 케라스 to_categorical.


**Q91.** ★ Dense(512, input_shape=(784,)) → Dense(512) → Dense(10) 모델의 층별 파라미터 수와 총합은?

- 정답: 784×512+512 = 401,920 / 512×512+512 = 262,656 / 512×10+10 = 5,130. 총 669,706.


**Q92.** 위 모델에서 Activation·Dropout 층의 파라미터 수는?

- 정답: 0개 (학습할 가중치가 없다).


**Q93.** Flatten(28,28) → Dense(128) → Dropout → Dense(10)에서 `model.get_weights()`의 w[0], w[1], w[3] shape는?

- 정답: (784,128), (128,), (10,). (w[2]는 (128,10))


**Q94.** Fashion MNIST의 클래스 수와, `plt.cm.binary`로 표시하면 어떻게 보이나?

- 정답: 10개(T-shirt \~ Ankle boot). 흰 바탕에 검은 물체(반전된 흑백).


**Q95.** ★ IMDB 리뷰를 pad_sequences(maxlen=256, padding="post")로 맞추는 이유는? 더 긴 리뷰와 짧은 리뷰는 어떻게 되나?

- 정답: 리뷰마다 길이가 달라서, 신경망에 한 번에 넣으려면 길이를 같게 맞춰야 한다. 256보다 길면 잘리고, 짧으면 뒤쪽을 0(\<PAD>)으로 채운다.


**Q96.** BMI 예제의 LabelEncoder는 fat/normal/thin을 어떤 숫자로 바꾸나?

- 정답: 알파벳 순서대로 fat=0, normal=1, thin=2.


**Q97.** BMI 모델 첫 Dense(512, input_shape=(2,))의 파라미터 수는?

- 정답: 2×512 + 512 = 1,536.


**Q98.** IMDB에서 `num_words=10000`과 `word_index` 값에 +3을 하는 이유는?

- 정답: 가장 자주 나오는 단어 1만 개만 쓴다. 0\~3을 \<PAD>, \<START>, \<UNK>, \<UNUSED> 특수 토큰에 주기 위해 3칸 밀어둔다.


**Q99.** ★ IMDB 모델 Embedding(10000,16) → GlobalAveragePooling1D → Dense(16) → Dense(1, sigmoid)의 파라미터 수는?

- 정답: 160,000 / 0 / 272 / 17. 총 160,289.


**Q100.** IMDB 모델의 loss와 출력 활성화 함수, 검증 데이터는 어떻게 만들었나?

- 정답: binary_crossentropy, sigmoid(긍정일 확률). 훈련 데이터 앞 10,000개를 검증으로 떼서 `validation_data`로 넣었다.


## 6. PyTorch·GPU (Q101\~120)


**Q101.** GPU를 쓸 수 있는지 확인하는 코드와 GPU 이름을 보는 코드는?

- 정답: `torch.cuda.is_available()`, `torch.cuda.get_device_name(0)`.


**Q102.** 그래픽 드라이버가 설치됐는지 확인하는 명령어는?

- 정답: cmd에서 `nvidia-smi`.


**Q103.** ★ GPU가 있으면 GPU, 없으면 CPU를 쓰는 device 코드는?

- 정답: `device = torch.device("cuda" if torch.cuda.is_available() else "cpu")`.


**Q104.** ★ GPU 학습에서 `.to(device)`로 옮겨야 하는 두 가지는? 하나만 옮기면?

- 정답: 모델과 데이터(입력·정답). 둘이 다른 장치에 있으면 에러가 난다.


**Q105.** ★ 파이토치에서 입력은 FloatTensor, 분류 정답은 LongTensor로 바꾸는 이유는?

- 정답: CrossEntropyLoss는 정답을 정수(클래스 번호, int64)로 받기 때문.


**Q106.** nn.Module을 상속한 클래스에서 반드시 작성하는 두 메서드는?

- 정답: `__init__`(층 정의, `super().__init__()` 호출)과 `forward`(데이터 흐름).


**Q107.** ★ IrisNet(Linear(4,16) → ReLU → Linear(16,3))의 파라미터 수는?

- 정답: 80 + 51 = 131.


**Q108.** ★ 파이토치 학습 루프 한 번의 4단계를 순서대로 쓰면?

- 정답: `optimizer.zero_grad()` → loss 계산 → `loss.backward()` → `optimizer.step()`.


**Q109.** ★ 파이토치에서 매 step마다 `optimizer.zero_grad()`를 해야 하는 이유는?

- 정답: 파이토치는 backward() 때 기울기를 덮어쓰지 않고 누적(더하기)한다. 비우지 않으면 이전 step의 기울기가 쌓여 업데이트가 점점 커지고 학습이 불안정해진다.


**Q110.** `model.train()`과 `model.eval()`의 차이는?

- 정답: 드롭아웃·배치정규화 같은 층의 동작 모드를 바꾼다. train은 드롭아웃 켬, eval은 끔. 가중치 학습 여부를 정하는 것은 아니다.


**Q111.** `torch.no_grad()`는 왜 쓰나?

- 정답: 평가·예측 때 기울기 계산을 꺼서 메모리와 시간을 아낀다.


**Q112.** `model(X).argmax(dim=1)`은 무엇을 돌려주나?

- 정답: 각 샘플에서 점수가 가장 큰 클래스 번호(예측 클래스).


**Q113.** ★ nn.CrossEntropyLoss를 쓸 때 모델 마지막에 Softmax를 넣나?

- 정답: 넣지 않는다. CrossEntropyLoss 안에 softmax가 포함돼 있어 출력(logit)을 그대로 넣는다.


**Q114.** BCELoss와 BCEWithLogitsLoss의 차이는?

- 정답: BCELoss는 모델 끝에 Sigmoid가 필요. BCEWithLogitsLoss는 Sigmoid가 내장돼 더 안정적.


**Q115.** TensorDataset과 DataLoader의 역할은?

- 정답: TensorDataset은 입력과 정답을 하나로 묶고, DataLoader는 미니배치로 나눠 준다(shuffle 가능).


**Q116.** 보스턴 집값의 정답을 `.view(-1, 1)`로 바꾸는 이유는?

- 정답: 모델 출력이 (N,1)이라 모양을 맞춰야 MSELoss가 제대로 계산된다.


**Q117.** 보스턴 데이터 506개를 8:1:1로 나눌 때 코드와 결과 크기는?

- 정답: train_test_split을 두 번(test_size=0.2 후 남은 것을 0.5). train 404, val 51, test 51.


**Q118.** `torch.manual_seed(42)`와 `torch.backends.cudnn.deterministic = True`를 설정하는 목적은?

- 정답: 재현성. 같은 입력에 대해 최대한 같은 결과가 나오도록 한다(완전히 강제되는 것은 아님).


**Q119.** ★ 훈련(train)·검증(validation)·테스트(test) 데이터의 역할은 각각 무엇인가?

- 정답: 훈련은 가중치 학습, 검증은 학습 중 과적합 확인·조기 종료·하이퍼파라미터 선택, 테스트는 마지막에 한 번 최종 성능 평가. 테스트 데이터는 학습에 어떤 식으로든 쓰면 안 된다.


**Q120.** 보스턴 모델 13 → 64 → 32 → 1의 파라미터 수는? MAE와 MSE의 차이는?

- 정답: 896 + 2,080 + 33 = 3,009. MAE는 오차 절댓값의 평균(원래 단위), MSE는 제곱 평균(큰 오차에 더 민감).


## 7. CNN (Q121\~150)


**Q121.** ★ 이미지를 MLP(Dense)로 처리할 때의 한계는?

- 정답: 1차원으로 펼치면 가까운 픽셀끼리의 관계 같은 공간적 구조 정보가 사라지고, 픽셀 몇 개만 달라도 예측이 크게 흔들린다.


**Q122.** 이미지 텐서의 3차원 구성과, 흑백·컬러의 채널 수는?

- 정답: (높이, 너비, 채널). 흑백 1, 컬러(RGB) 3. 채널은 깊이(depth)라고도 한다.


**Q123.** 합성곱 연산이란?

- 정답: 커널(필터)을 이미지 왼쪽 위부터 순서대로 옮기며, 겹치는 원소끼리 곱해 모두 더한 값을 출력하는 것. 결과를 특성 맵(Feature Map)이라 한다.


**Q124.** ★ 아래 이미지 영역과 커널의 합성곱 결과는?

- 영역 [[1,2,3],[2,1,0],[3,0,1]], 커널 [[1,0,1],[1,0,1],[0,1,0]]
- 정답: 1+0+3+2+0+0+0+0+0 = 6.


**Q125.** 일반적으로 쓰는 커널 크기는? 작은 필터를 중첩하는 이유는?

- 정답: 3×3 또는 5×5(요즘은 대부분 3×3). 특징을 더 돋보이게 하면서 연산량을 줄인다.


**Q126.** 스트라이드(stride)란? 커지면 출력은?

- 정답: 커널의 이동 간격. 커질수록 특성 맵이 작아진다.


**Q127.** ★ 특성 맵 크기 공식(패딩 P 포함)은?

- 정답: O = (I − K + 2P) / S + 1 (소수점 버림).


**Q128.** ★ 5×5 이미지, 3×3 커널에서 스트라이드 1과 2일 때 특성 맵 크기는?

- 정답: 3×3(9스텝), 2×2.


**Q129.** 패딩을 쓰는 이유와 주로 채우는 값은?

- 정답: 합성곱 후에도 크기를 유지하기 위해. 0(제로 패딩).


**Q130.** 스트라이드 1에서 크기를 유지하려면 3×3, 5×5 커널에 패딩 폭을 얼마로?

- 정답: 1, 2.


**Q131.** Conv2D의 padding "valid"와 "same"의 차이는?

- 정답: valid는 패딩 없음(출력이 작아짐, 기본값), same은 출력 크기 = 입력 크기.


**Q132.** ★ 3×3 이미지를 MLP(은닉 4개)로 처리할 때와, 2×2 커널 CNN으로 처리할 때 가중치 수는?

- 정답: MLP 9×4 = 36개, CNN 4개(w0\~w3). CNN은 같은 커널을 이미지 전체에 공유한다.


**Q133.** CNN에서 편향은 몇 개이고 어디에 더해지나?

- 정답: 필터 하나당 1개. 커널을 적용한 결과의 모든 원소에 똑같이 더한다.


**Q134.** ★ 입력 채널이 3개일 때 커널의 채널 수는? 출력 특성 맵의 채널은?

- 정답: 커널도 3채널(3채널짜리 커널 1개). 채널별 결과를 모두 더해 출력은 1채널.


**Q135.** 합성곱 층에서 주로 쓰는 활성화 함수는?

- 정답: ReLU(또는 변형).


**Q136.** ★ 풀링의 역할과 종류 세 가지는?

- 정답: 특성 맵을 다운샘플링해 크기를 줄임. 최대(max)·평균(average)·최소(min) 풀링.


**Q137.** ★ 풀링과 합성곱의 차이 두 가지는?

- 정답: 풀링은 ① 학습할 가중치가 없고, ② 연산 후 채널 수가 변하지 않는다.


**Q138.** 2×2 맥스풀링(스트라이드 2) 영역 [[1,3],[2,8]]의 결과는? 28×28에 적용하면?

- 정답: 8. 14×14.


**Q139.** CNN 전체 구조의 두 단계와 각 단계의 층은?

- 정답: ① 특징 추출(Conv + Pooling 반복), ② 분류(Flatten → FC → Softmax). Flatten은 shape만 바꾼다.


**Q140.** 풀링을 지나면 출력이 1/4로 줄어든다. 다음 Conv의 필터 수는 어떻게 정하는 것이 좋나?

- 정답: 특성 맵 수를 늘려(자료 기준 약 4배) 층마다 연산량을 비슷하게 유지한다. 실습에서는 16 → 32, 32 → 64처럼 2배씩 늘렸다.


**Q141.** 수평 필터 [[1,1,1],[0,0,0],[−1,−1,−1]]은 무엇을 검출하나? 수평·수직 결과를 합치는 소벨 식은?

- 정답: 가로 경계선. sqrt(수평² + 수직²).


**Q142.** CNN은 특징을 사람이 직접(수동으로) 추출해야 하나? 어떤 데이터에 특히 유용한가?

- 정답: 아니다. 데이터에서 직접 학습하며, 데이터 전체 대신 지역적인 부분에서 특징을 뽑아 Feature Map을 만든다. 이미지에서 객체·얼굴·장면을 인식하는 데 특히 유용하다.


**Q143.** ★ Keras에서 28×28 MNIST를 Conv2D에 넣기 전에 Reshape((28,28,1))을 하는 이유는?

- 정답: Conv2D는 (높이, 너비, 채널) 3차원 입력을 받기 때문. 흑백이라 채널 1을 추가한다.


**Q144.** ★ Conv2D(16, 3) 입력 (28,28,1)의 출력 shape와 파라미터 수는?

- 정답: (26,26,16), 16 × (3×3×1) + 16 = 160.


**Q145.** ★ Conv2D(16,3) → MaxPool(2) → Conv2D(32,3) → MaxPool(2) → Flatten의 단계별 shape와, 두 번째 Conv의 파라미터 수는?

- 정답: 26×26×16 → 13×13×16 → 11×11×32 → 5×5×32 → 800. 파라미터 32 × (3×3×16) + 32 = 4,640.


**Q146.** 위 모델을 padding="same"으로 바꾸면 Flatten 크기는?

- 정답: 28 → 14 → 14 → 7, 7×7×32 = 1,568.


**Q147.** ★ CIFAR-10 모델 Conv(32) → Pool → Conv(64) → Pool → Conv(64) → Flatten에서 첫 Conv 파라미터 수와 Flatten 크기는? (입력 32×32×3)

- 정답: 32 × (3×3×3) + 32 = 896. 30 → 15 → 13 → 6 → 4, 4×4×64 = 1,024.


**Q148.** Sequential과 Functional API의 차이는?

- 정답: Functional은 `keras.Input`으로 시작해 `x = layer(...)(x)`처럼 함수 호출하듯 연결하고 `keras.Model(inputs, outputs)`로 만든다. 분기·다중 입력 같은 복잡한 구조에 쓴다.


**Q149.** ★ 파이토치 CNN에서 `reshape(-1, 1, 28, 28)`으로 만드는 이유와, Conv2d(1,32,3,same) → Pool → Conv2d(32,64,3,same) → Pool → Flatten 크기는?

- 정답: 파이토치는 (N, 채널, 높이, 너비) 순서(케라스는 채널이 마지막). 7×7×64 = 3,136.


**Q150.** ★ 파이토치 CNN의 조기 종료(patience=2)는 몇 에포크에서 멈췄고, 저장된 모델은 몇 번째 에포크 것인가?

- 정답: 검증손실이 8에포크 0.2200에서 최저, 9·10에포크 연속으로 더 좋아지지 않아 10에포크에서 종료. 저장된 것은 8에포크 모델(`load_state_dict`로 불러옴). 테스트 정확도 0.9135.


## 8. RNN·LSTM·양방향 RNN·텍스트 생성 (Q151\~175)


**Q151.** ★ FFNN과 비교한 RNN의 핵심 차이는?

- 정답: 은닉층의 결과가 다시 같은 은닉층의 입력으로 들어가, 순서(시간)를 고려할 수 있다. 문장·음성·주가 같은 시퀀스 데이터에 적합.


**Q152.** 메모리 셀이 출력층과 다음 시점의 자신에게 보내는 값을 무엇이라 하나?

- 정답: 은닉 상태(hidden state).


**Q153.** ★ RNN 은닉 상태 식과 주로 쓰는 활성화 함수는?

- 정답: hₜ = tanh(Wₓxₜ + Wₕhₜ₋₁ + b). 활성화 함수는 주로 tanh(하이퍼볼릭탄젠트).


**Q154.** RNN의 가중치 Wₓ, Wₕ, Wᵧ는 시점마다 다른가?

- 정답: 모든 시점에서 같은 값을 공유한다. 단, 은닉층이 2개면 층끼리는 다른 가중치.


**Q155.** ★ 일대다 / 다대일 / 다대다의 대표 예는?

- 정답: 이미지 캡셔닝 / 감성 분류·스팸 판별 / 챗봇·번역·개체명 인식·품사 태깅.


**Q156.** ★ `SimpleRNN(3, input_shape=(2, 10))`에서 3, 2, 10은 각각 무엇인가?

- 정답: 3 = 은닉 상태 크기(hidden_size), 2 = timesteps(시퀀스 길이), 10 = input_dim(입력 벡터 크기).


**Q157.** ★ 위 SimpleRNN의 파라미터 수는? 공식은?

- 정답: units × (input_dim + units + 1) = 3 × (10 + 3 + 1) = 42.


**Q158.** ★ 입력 (8, 2, 10)일 때 return_sequences=False와 True의 출력 shape는?

- 정답: False → (8, 3) 2D, 마지막 시점만. True → (8, 2, 3) 3D, 모든 시점.


**Q159.** RNN을 2층 쌓을 때 첫 층에 return_sequences=True가 필요한 이유는?

- 정답: 다음 RNN 층은 시퀀스(3D) 입력을 받기 때문. 모든 시점의 은닉 상태를 넘겨야 한다.


**Q160.** 신버전 케라스에서 input_shape 없이 SimpleRNN(3)만 넣고 summary를 보려면?

- 정답: `model(inputs)`처럼 실제 데이터를 한 번 넣거나 `model.build(input_shape=...)`를 호출해 모델을 빌드한다.


**Q161.** ★ 장기 의존성 문제란?

- 정답: 시점이 길어질수록 앞쪽 정보가 뒤로 충분히 전달되지 못하는 문제. (예: "모스크바에 여행을 왔는데 ... 여기 ___")


**Q162.** ★ LSTM의 세 게이트와, 게이트에 공통으로 들어가는 함수는?

- 정답: 입력 게이트, 삭제(망각) 게이트, 출력 게이트. 시그모이드(0\~1 값으로 열고 닫기 조절).


**Q163.** 입력 게이트에서 나오는 iₜ와 gₜ는 각각 어떤 함수를 지나나?

- 정답: iₜ는 시그모이드(0\~1), gₜ는 tanh(−1\~1). 둘을 원소별 곱해 이번에 기억할 값을 정한다.


**Q164.** 삭제 게이트 값이 0에 가까우면? 1에 가까우면?

- 정답: 0이면 이전 기억을 많이 지운다. 1이면 온전히 기억한다.


**Q165.** ★ LSTM에서 장기 상태와 단기 상태는 각각 무엇인가?

- 정답: 장기 상태 = 셀 상태 Cₜ, 단기 상태 = 은닉 상태 hₜ(Cₜ를 tanh에 통과시킨 뒤 출력 게이트와 곱한 값).


**Q166.** 삭제 게이트 fₜ=0이면 Cₜ는 무엇으로 결정되나? iₜ=0이면?

- 정답: fₜ=0이면 입력 게이트 결과만으로, iₜ=0이면 이전 셀 상태 Cₜ₋₁만으로 결정된다.


**Q167.** ★ 양방향 RNN의 아이디어와 구조는?

- 정답: 이전 시점뿐 아니라 이후 시점의 입력도 예측에 도움이 된다. 앞→뒤(Forward)와 뒤→앞(Backward) 메모리 셀 두 개를 쓰고 두 값을 모두 출력에 쓴다.


**Q168.** ★ `Bidirectional(SimpleRNN(3, return_sequences=True), input_shape=(10, 5))`의 출력 shape와 파라미터 수는?

- 정답: (None, 10, 6) — 양방향이라 3×2. 파라미터 2 × 3 × (5+3+1) = 54.


**Q169.** ★ "경마장에 있는 말이 뛰고 있다 / 그의 말이 법이다 / 가는 말이 고와야 오는 말이 곱다"의 vocab_size는? 왜 +1을 하나?

- 정답: 고유 단어 11개 + 1 = 12. Tokenizer 인덱스가 1부터 시작하고 0은 패딩용이라, 원핫 크기를 맞추려고 1을 더한다.


**Q170.** 위 문장으로 만든 학습 샘플 수와 max_len은? x와 y(원핫 후)의 shape는?

- 정답: 4+2+5 = 11개, max_len 6. x (11, 5), y (11, 12).


**Q171.** 텍스트 생성에서 pad_sequences를 padding="pre"로 하는 이유는?

- 정답: 마지막 값을 예측 정답(y)으로 쓰기 때문. 앞을 0으로 채워야 실제 단어가 끝에 모이고 `sequences[:, -1]`이 다음 단어가 된다.


**Q172.** Embedding(12, 6) → SimpleRNN(32) → Dense(12)의 층별 파라미터 수는?

- 정답: 12×6 = 72 / 32 × (6+32+1) = 1,248 / 32×12+12 = 396.


**Q173.** ★ LSTM(128)의 입력이 10차원 임베딩일 때 파라미터 수는? SimpleRNN과 비교하면?

- 정답: 4 × 128 × (10+128+1) = 71,168. 입력·삭제·출력 게이트와 gₜ, 모두 4개가 각자 가중치를 가져서 SimpleRNN의 4배.


**Q174.** sentence_generation("체포하겠어", 4)처럼 학습에 없던 단어를 넣으면 어떻게 되나?

- 정답: texts_to_sequences가 빈 리스트를 내서 입력이 전부 0(패딩)이 된다. 모델은 학습된 패턴대로 아무 단어나 이어 붙인다.


**Q175.** ★ Embedding 층의 역할과, `Embedding(input_dim, output_dim)`의 두 인자는 무엇인가?

- 정답: 단어 번호(정수)를 일정한 크기의 벡터로 바꾼다. input_dim은 단어 집합 크기(vocab_size), output_dim은 단어 하나를 표현할 벡터 차원(예: 16, 6, 10).


## 9. 시계열·평활법·보간법·Prophet·ARIMA (Q176\~200)


**Q176.** 시계열 데이터 처리 5단계를 순서대로 쓰면?

- 정답: 정제 → 변환(로그·차분) → 분석(추세·계절성) → 모델링(ARIMA·Prophet) → 검증.


**Q177.** ★ 엄격한 정상성과 약한 정상성의 차이는?

- 정답: 엄격한 정상성은 확률분포 자체가 시간에 따라 변하지 않음. 약한 정상성은 평균과 시차별 자기공분산만 일정하면 됨.


**Q178.** ★ 비정상 시계열의 특징 세 가지는?

- 정답: ① 추세나 계절성이 보인다, ② 평균·분산이 시간에 따라 변한다, ③ ACF가 천천히 감소하고 시차 1의 ACF가 큰 양수다.


**Q179.** 구글 주가(가격 자체)와 구글 주가의 일일 변동 중 정상 시계열에 가까운 것은?

- 정답: 일일 변동(차분한 값). 가격 자체는 추세가 있어 비정상.


**Q180.** 시계열 데이터의 네 가지 형태는?

- 정답: 우연변동, 계절변동, 추세변동, 계절적 추세변동 시계열.


**Q181.** ★ 시계열 분해의 세 요소와, 계절성과 주기성(cycle)의 차이는?

- 정답: 계절성, 추세, 불규칙 요소(residual). 주기가 고정되면 계절성, 고정되지 않으면 주기성.


**Q182.** 추세·계절성을 없애는 세 방법과, 노이즈가 자주/가끔 생길 때 각각 알맞은 방법은?

- 정답: 회귀분석, 평활법, 차분. 자주 → 평활법, 가끔 → 차분.


**Q183.** ★ 1차 차분과 계절성 차분의 식과 각각 무엇을 없애나?

- 정답: Δyₜ = yₜ − yₜ₋₁(추세 제거), Δyₜ = yₜ − yₜ₋ₘ(계절성 제거, m=주기).


**Q184.** `np.diff([1,2,3,4,5])`의 결과와 길이는? 2차 이상 차분을 조심해야 하는 이유는?

- 정답: [1,1,1,1], 길이 4. 차분할수록 데이터가 줄어 설명력이 낮아진다. 보통 1차 차분으로 충분하다.


**Q185.** 차분을 하면 시계열이 정상성을 갖게 되는 이유는?

- 정답: 비정상 시계열은 대부분 값이 누적되는 누적과정(Integrated Process)이라, 누적된 값을 빼면 정상 시계열이 된다.


**Q186.** 로그 변환의 역할과 한계는?

- 정답: 분산을 안정시킨다. 로그만으로는 추세가 완전히 없어지지 않아 차분을 같이 쓴다.


**Q187.** ★ 자기상관 함수(ACF)란? 시계열 자료가 자기상관을 갖는다는 것은 무슨 뜻인가?

- 정답: 시차(lag)에 따른 시계열의 자기상관을 나타내는 함수. 오늘 주가가 어제 주가의 영향을 받고 내일 주가에 영향을 주듯, 과거 값과 현재 값이 서로 관련돼 있다는 뜻.


**Q188.** ★ `WMA([10, 20, 30, 40], 3)`(가중치 1,2,3)의 결과는? SMA(window=10)의 앞 몇 개가 NaN인가?

- 정답: [NaN, NaN, 23.33, 33.33]. (10·1+20·2+30·3)/6 = 23.33. SMA는 앞 9개가 NaN.


**Q189.** 시계열 결측치를 모델링으로 채우기 어려운 이유와, 최근 등장한 대안은?

- 정답: 시계열은 시간에 따라 평균과 분산이 달라서 모델을 만들기에 데이터가 부족할 가능성이 높다. 최근에는 GAN으로 시계열 데이터를 생성해 대체하려는 시도가 있다.


**Q190.** ★ 이동평균 평활법과 지수 평활법의 차이는?

- 정답: 이동평균은 일정 구간(window)의 평균. 지수 평활법은 모든 과거 자료를 쓰되 최근 값일수록 더 큰 가중치를 준다.


**Q191.** SimpleExpSmoothing의 smoothing_level(α)을 0.2에서 0.5로 키우면? `fit()`에 값을 안 주면?

- 정답: 최근 값을 더 많이 반영해 원래 데이터를 더 빨리 따라간다(덜 부드러움). 값을 안 주면 최적 α를 자동으로 추정한다.


**Q192.** ★ 시계열 결측치 처리 방법 LOCF와 NOCB는?

- 정답: LOCF는 직전 관측값, NOCB는 직후 관측값으로 채운다. (그 외 이동평균·중앙값, 보간법, 모델링)


**Q193.** ★ 선형 보간에서 A=[1, 2, NaN, 4, 5], B=[5, NaN, NaN, 8, 9]는 어떻게 채워지나?

- 정답: A의 NaN → 3.0, B의 NaN → 6.0, 7.0.


**Q194.** 선형·다항식·스플라인 보간의 차이는?

- 정답: 선형은 두 점을 직선으로, 다항식은 전체를 다항식 곡선으로, 스플라인은 소구간마다 다항식을 이어 부드러운 곡선으로 추정. 급격한 변동이 있으면 스플라인.


**Q195.** ★ Prophet에 넣는 데이터프레임의 필수 컬럼 이름은?

- 정답: ds(날짜, datetime)와 y(값).


**Q196.** ★ Prophet의 구성 요소 네 가지와, 계절성을 근사하는 수학 도구는?

- 정답: g(t) 추세, s(t) 계절성, h(t) 휴일 효과, εₜ 오차(정규분포 가정). 계절성은 푸리에 급수.


**Q197.** Prophet의 make_future_dataframe(periods=365)와 predict로 얻는 주요 컬럼 yhat, yhat_lower, yhat_upper는?

- 정답: 기존 날짜 + 미래 365일의 ds를 만든다. yhat 예측값, yhat_lower 하한, yhat_upper 상한. 월별이면 `freq="MS"`, 미래만 원하면 `include_history=False`.


**Q198.** ★ ARIMA(p, d, q)의 세 문자는 각각 무엇인가?

- 정답: p = AR(과거 몇 시점의 값을 쓸지), d = I(몇 번 차분할지), q = MA(과거 몇 개의 오차항을 쓸지).


**Q199.** ★ ARIMA 모델의 특징 세 가지와 활용 분야는?

- 정답: ① 비정상 시계열에도 적용 가능(차분으로 정상성을 확보한 뒤 AR·MA 적용), ② 파라미터 조정으로 다양한 패턴을 모델링하는 유연성, ③ 단기·장기 예측 모두 가능. 활용 분야는 경제·금융, 기상학, 수요 예측.


**Q200.** ★ 시계열 데이터를 train/test로 나눌 때 주의할 점은? 삼성전자·자동차 판매량 예제는 어떻게 나눴나?

- 정답: 섞지 않고 시간 순서대로 앞부분을 학습, 뒷부분을 테스트로 쓴다(미래로 과거를 맞히면 누수). 삼성전자는 앞 80% / 뒤 20%, 자동차 판매량은 마지막 12개월을 테스트로 두고 MAE로 평가했다.


## 10. 파라미터·출력 크기 계산 20문제

### 공식 모음

| 층 | 파라미터 수 | 출력 크기 |
| --- | --- | --- |
| Dense | (입력 수 + 1) × 뉴런 수 | 뉴런 수 |
| Conv2D | (커널 가로 × 커널 세로 × **입력 채널** + 1) × 필터 수 | (입력 − 커널 + 2×패딩) ÷ 스트라이드 + 1, 채널 = 필터 수 |
| MaxPooling2D | 0 | 가로·세로 ÷ 2 (소수점 버림), 채널 그대로 |
| Flatten · Dropout · Activation | 0 | Flatten은 전부 곱한 값 |
| Embedding | 단어 수 × 벡터 차원 (편향 없음) | (배치, 문장 길이, 벡터 차원) |
| SimpleRNN | units × (입력 차원 + units + 1) | False면 (배치, units), True면 (배치, 시점, units) |
| LSTM | 4 × units × (입력 차원 + units + 1) | SimpleRNN과 같음 |
| Bidirectional | 안쪽 층 × 2 | units × 2 |

**체크리스트:** ① 편향(+1) 더했나? ② 두 번째 Conv부터 입력 채널 = 앞 층 필터 수? ③ 풀링은 0개?

아래 확인 코드는 TensorFlow 2.x에서 실행하면 `model.summary()`로 답을 직접 확인할 수 있어요.


In [ ]:
from tensorflow import keras
from tensorflow.keras import layers


**P1. Dense 층 (MNIST 은닉층)** — 입력 784개를 받는 `Dense(256)` 층의 파라미터 수는?

**풀이**

(784 + 1) × 256 = 785 × 256 = **200,960**

가중치 784 × 256 = 200,704, 편향 256개.


In [ ]:
m = keras.Sequential([keras.Input(shape=(784,)), layers.Dense(256)])
m.summary()


**P2. Dense 층 (회귀 출력층)** — 입력 100개를 받는 회귀 출력층 `Dense(1)`의 파라미터 수는?

**풀이**

(100 + 1) × 1 = **101**

가중치 100개, 편향 1개.


In [ ]:
m = keras.Sequential([keras.Input(shape=(100,)), layers.Dense(1)])
m.summary()


**P3. MLP 전체 (Iris)** — 입력 4 → `Dense(16)` → `Dense(3)` 모델의 전체 파라미터 수는?

**풀이**

(4+1)×16 = 80, (16+1)×3 = 51 → 합계 **131**


In [ ]:
m = keras.Sequential([keras.Input(shape=(4,)), layers.Dense(16, activation='relu'), layers.Dense(3, activation='softmax')])
m.summary()


**P4. MLP 전체 (보스턴 집값)** — 입력 13 → `Dense(64)` → `Dense(32)` → `Dense(1)` 모델의 전체 파라미터 수는?

**풀이**

(13+1)×64 = 896, (64+1)×32 = 2,080, (32+1)×1 = 33 → 합계 **3,009**


In [ ]:
m = keras.Sequential([keras.Input(shape=(13,)), layers.Dense(64, activation='relu'), layers.Dense(32, activation='relu'), layers.Dense(1)])
m.summary()


**P5. Conv2D 흑백** — (28, 28, 1) 입력에 `Conv2D(8, (3,3))` (패딩 없음, 스트라이드 1)을 적용할 때 출력 shape와 파라미터 수는?

**풀이**

- 출력: 28 − 3 + 1 = 26 → **(26, 26, 8)**
- 파라미터: (3×3×1 + 1) × 8 = 10 × 8 = **80**


In [ ]:
m = keras.Sequential([keras.Input(shape=(28,28,1)), layers.Conv2D(8, (3,3))])
m.summary()


**P6. Conv2D 5×5 커널** — (28, 28, 1) 입력에 `Conv2D(32, (5,5))`를 적용할 때 출력 shape와 파라미터 수는?

**풀이**

- 출력: 28 − 5 + 1 = 24 → **(24, 24, 32)**
- 파라미터: (5×5×1 + 1) × 32 = 26 × 32 = **832**


In [ ]:
m = keras.Sequential([keras.Input(shape=(28,28,1)), layers.Conv2D(32, (5,5))])
m.summary()


**P7. Conv2D 컬러** — (32, 32, 3) 컬러 입력에 `Conv2D(64, (3,3))`을 적용할 때 출력 shape와 파라미터 수는?

**풀이**

- 출력: 32 − 3 + 1 = 30 → **(30, 30, 64)**
- 파라미터: (3×3×**3** + 1) × 64 = 28 × 64 = **1,792**

입력 채널이 3(RGB)이라 필터 한 장이 3×3×3 = 27칸이에요.


In [ ]:
m = keras.Sequential([keras.Input(shape=(32,32,3)), layers.Conv2D(64, (3,3))])
m.summary()


**P8. Conv2D 두 번째 층 + same 패딩** — (14, 14, 16) 특성 맵에 `Conv2D(32, (3,3), padding='same')`을 적용할 때 출력 shape와 파라미터 수는?

**풀이**

- 출력: same이라 크기 유지 → **(14, 14, 32)**
- 파라미터: (3×3×**16** + 1) × 32 = 145 × 32 = **4,640**

두 번째 Conv부터 입력 채널 = 앞 층의 필터 수(16).


In [ ]:
m = keras.Sequential([keras.Input(shape=(14,14,16)), layers.Conv2D(32, (3,3), padding='same')])
m.summary()


**P9. Conv2D 스트라이드 2** — (28, 28, 1) 입력에 `Conv2D(16, (3,3), strides=2)` (패딩 없음)를 적용할 때 출력 shape와 파라미터 수는?

**풀이**

- 출력: (28 − 3) ÷ 2 + 1 = 12.5 + 1 → 소수점 버림 → **13** → **(13, 13, 16)**
- 파라미터: (3×3×1 + 1) × 16 = **160** (스트라이드는 파라미터 수에 영향 없음)


In [ ]:
m = keras.Sequential([keras.Input(shape=(28,28,1)), layers.Conv2D(16, (3,3), strides=2)])
m.summary()


**P10. MaxPooling** — (24, 24, 32) 특성 맵에 `MaxPooling2D((2,2))`를 적용할 때 출력 shape와 파라미터 수는?

**풀이**

- 출력: 가로·세로만 절반 → **(12, 12, 32)** (채널 그대로)
- 파라미터: **0** (풀링은 가중치 없음)


In [ ]:
m = keras.Sequential([keras.Input(shape=(24,24,32)), layers.MaxPooling2D((2,2))])
m.summary()


**P11. Flatten → Dense** — (5, 5, 64) 특성 맵을 `Flatten()` 한 뒤 `Dense(128)`에 넣을 때, Flatten 크기와 Dense 파라미터 수는?

**풀이**

- Flatten: 5 × 5 × 64 = **1,600** (Flatten 자체 파라미터 0)
- Dense: (1,600 + 1) × 128 = **204,928**


In [ ]:
m = keras.Sequential([keras.Input(shape=(5,5,64)), layers.Flatten(), layers.Dense(128)])
m.summary()


**P12. CNN 전체 (패딩 없음)** — 입력 (28,28,1) → `Conv2D(32,(3,3))` → `MaxPooling2D(2)` → `Conv2D(64,(3,3))` → `MaxPooling2D(2)` → `Flatten` → `Dense(10)`의 층별 출력 shape와 전체 파라미터 수는?

**풀이**

| 층 | 출력 shape | 파라미터 |
| --- | --- | --- |
| Conv2D(32) | (26, 26, 32) | (9×1+1)×32 = 320 |
| MaxPool | (13, 13, 32) | 0 |
| Conv2D(64) | (11, 11, 64) | (9×32+1)×64 = 18,496 |
| MaxPool | (5, 5, 64) | 0 |
| Flatten | 1,600 | 0 |
| Dense(10) | 10 | 1,601×10 = 16,010 |

전체 **34,826**. 11 ÷ 2 = 5.5 → 소수점 버림 5.


In [ ]:
m = keras.Sequential([keras.Input(shape=(28,28,1)), layers.Conv2D(32,(3,3),activation='relu'), layers.MaxPooling2D(2), layers.Conv2D(64,(3,3),activation='relu'), layers.MaxPooling2D(2), layers.Flatten(), layers.Dense(10, activation='softmax')])
m.summary()


**P13. CNN 전체 (same 패딩)** — 위 12번 모델에서 두 Conv2D를 모두 `padding='same'`으로 바꾸면 Flatten 크기와 전체 파라미터 수는?

**풀이**

28 → 28 → 14 → 14 → 7, Flatten = 7 × 7 × 64 = **3,136**

- Conv 파라미터는 그대로: 320 + 18,496
- Dense(10): 3,137 × 10 = 31,370
- 전체 **50,186**

패딩은 Conv 파라미터 수는 안 바꾸지만, 뒤 Dense 파라미터를 바꿔요.


In [ ]:
m = keras.Sequential([keras.Input(shape=(28,28,1)), layers.Conv2D(32,(3,3),padding='same',activation='relu'), layers.MaxPooling2D(2), layers.Conv2D(64,(3,3),padding='same',activation='relu'), layers.MaxPooling2D(2), layers.Flatten(), layers.Dense(10, activation='softmax')])
m.summary()


**P14. SimpleRNN** — 입력 차원 8인 시퀀스를 `SimpleRNN(16)`에 넣을 때 파라미터 수는?

**풀이**

units × (입력 차원 + units + 1) = 16 × (8 + 16 + 1) = 16 × 25 = **400**

(입력→은닉 8×16, 은닉→은닉 16×16, 편향 16)


In [ ]:
m = keras.Sequential([keras.Input(shape=(None, 8)), layers.SimpleRNN(16)])
m.summary()


**P15. Embedding + SimpleRNN + Dense** — `Embedding(1000, 8)` → `SimpleRNN(32)` → `Dense(1, sigmoid)` 모델의 층별 파라미터와 합계는?

**풀이**

- Embedding: 1,000 × 8 = 8,000
- SimpleRNN: 32 × (8 + 32 + 1) = 1,312
- Dense: (32 + 1) × 1 = 33
- 합계 **9,345**

SimpleRNN의 입력 차원 = Embedding의 출력 차원(8).


In [ ]:
m = keras.Sequential([keras.Input(shape=(None,)), layers.Embedding(1000, 8), layers.SimpleRNN(32), layers.Dense(1, activation='sigmoid')])
m.summary()


**P16. LSTM** — 입력 차원 32인 시퀀스를 `LSTM(64)`에 넣을 때 파라미터 수는?

**풀이**

4 × units × (입력 차원 + units + 1) = 4 × 64 × (32 + 64 + 1) = 4 × 64 × 97 = **24,832**

LSTM은 입력·삭제·출력 게이트와 g, 4묶음이라 SimpleRNN의 4배.


In [ ]:
m = keras.Sequential([keras.Input(shape=(None, 32)), layers.LSTM(64)])
m.summary()


**P17. LSTM 작은 예** — 입력 차원 5인 시퀀스를 `LSTM(16)`에 넣을 때 파라미터 수는? 같은 크기의 SimpleRNN(16)이면?

**풀이**

- LSTM: 4 × 16 × (5 + 16 + 1) = 4 × 352 = **1,408**
- SimpleRNN: 16 × 22 = **352** (정확히 4분의 1)


In [ ]:
m = keras.Sequential([keras.Input(shape=(None, 5)), layers.LSTM(16)])
m.summary()
m2 = keras.Sequential([keras.Input(shape=(None, 5)), layers.SimpleRNN(16)])
m2.summary()


**P18. 양방향 LSTM** — 입력 차원 10인 시퀀스를 `Bidirectional(LSTM(32))`에 넣을 때 출력 크기와 파라미터 수는?

**풀이**

- 출력: 앞→뒤 32 + 뒤→앞 32 = **64**
- 파라미터: LSTM 하나 4 × 32 × (10 + 32 + 1) = 5,504, 두 방향이라 × 2 = **11,008**


In [ ]:
m = keras.Sequential([keras.Input(shape=(None, 10)), layers.Bidirectional(layers.LSTM(32))])
m.summary()


**P19. Embedding** — 길이 100으로 패딩된 문장을 `Embedding(5000, 32)`에 넣을 때 출력 shape와 파라미터 수는?

**풀이**

- 출력: 단어 100개가 각각 32차원 벡터 → **(None, 100, 32)**
- 파라미터: 5,000 × 32 = **160,000** (편향 없음, 단어마다 벡터 하나씩 저장하는 표)


In [ ]:
m = keras.Sequential([keras.Input(shape=(100,)), layers.Embedding(5000, 32)])
m.summary()


**P20. RNN 2층 쌓기** — 입력 (None, 7, 4) → `SimpleRNN(10, return_sequences=True)` → `SimpleRNN(5)`의 층별 출력 shape와 파라미터 수는?

**풀이**

| 층 | 출력 shape | 파라미터 |
| --- | --- | --- |
| SimpleRNN(10, True) | (None, 7, 10) | 10 × (4 + 10 + 1) = 150 |
| SimpleRNN(5) | (None, 5) | 5 × (10 + 5 + 1) = 80 |

합계 **230**. 두 번째 층의 입력 차원 = 첫 층 units(10). 첫 층은 다음 RNN에 시퀀스를 넘겨야 해서 return_sequences=True.


In [ ]:
m = keras.Sequential([keras.Input(shape=(7, 4)), layers.SimpleRNN(10, return_sequences=True), layers.SimpleRNN(5)])
m.summary()


① Dense

입력이 전부 각 뉴런에 연결돼요.

(입력 수 + 1) × 뉴런 수

예: 입력 784 → Dense(128): (784 + 1) × 128 = 100,480

② Conv2D: 필터 한 장이 "뉴런 하나"

필터(돋보기) 한 장이 한 번에 보는 칸 수가 곧 "입력 개수"예요.

3×3 필터, 흑백(채널 1): 한 번에 3×3×1 = 9칸
3×3 필터, 컬러(채널 3): 돋보기가 RGB 세 장을 동시에 겹쳐 보니까 3×3×3 = 27칸

(커널 가로 × 커널 세로 × 입력 채널 + 1) × 필터 수

왜 사진 크기(28×28)는 공식에 없을까요?
같은 돋보기 하나를 사진 전체에 옮겨 가며 재사용하기 때문이에요(가중치 공유). 사진이 커져도 돋보기는 그대로예요.

가장 많이 틀리는 곳: 두 번째 Conv의 입력 채널은 앞 Conv의 필터 수예요.
Conv2D(16) 다음 Conv2D(32, 3)이면 (3×3×16 + 1) × 32 = 4,640

③ 파라미터가 0인 층

가중치 없이 정해진 동작만 하는 층들이에요.

MaxPooling: 가장 큰 값만 고름
Flatten: 모양만 펼침
Dropout: 일부를 끔

전부 0개

④ Embedding: 단어 사전 표

단어 번호마다 벡터 하나씩 저장한 표예요. 계산을 하는 층이 아니라 찾아보는 층이라 편향이 없어요.

단어 수 × 벡터 차원 (+1 없음)

예: Embedding(5000, 32) → 5,000 × 32 = 160,000

⑤ SimpleRNN: 입력이 두 군데서 들어옴

RNN은 "지금 들어온 단어"와 함께 “방금 전 내 기억(은닉 상태)”도 다시 받아요. 앞 문장을 기억하며 읽는 거죠.
그래서 뉴런 하나가 받는 입력은 **입력 차원 + units(내 기억 크기)**예요.

units × (입력 차원 + units + 1)

Dense 공식과 비교해 보면 입력 개수에 units만 더해진 거예요.

예: 입력 8, SimpleRNN(16) → 16 × (8 + 16 + 1) = 400

⑥ LSTM = SimpleRNN × 4

LSTM 안에는 SimpleRNN 같은 계산 묶음이 4개(입력 게이트, 삭제 게이트, 출력 게이트, g) 있어요.

4 × units × (입력 차원 + units + 1)

⑦ Bidirectional = × 2

앞→뒤로 읽는 RNN과 뒤→앞으로 읽는 RNN을 따로 하나씩 둬요.

파라미터 × 2, 출력 크기도 units × 2

2부. 출력 크기: 모양 따라가기
① Conv2D 출력 크기

돋보기를 사진 위에서 몇 번 옮길 수 있는지 세는 거예요.

O = (I − K + 2P) ÷ S + 1 (소수점 버림)
I 입력 크기, K 커널 크기, P 패딩, S 스트라이드

이해하는 법(스트라이드 1일 때):

5칸짜리 줄에 3칸 돋보기를 놓으면, 놓을 수 있는 자리는 1~3, 2~4, 3~5로 3군데예요 → 5 − 3 + 1 = 3
패딩: 테두리에 0을 둘러서 길을 넓혀요 → 양쪽이니까 + 2P
스트라이드: 한 번에 몇 칸씩 건너뛸지 → 그만큼 나눠요

외우면 편한 것:

valid(기본값, 패딩 없음): I − K + 1 (3×3이면 2 줄어듦)
same: 크기 그대로
채널 = 필터 수
② MaxPooling(2)

가로·세로 ÷ 2, 소수점 버림, 채널은 그대로

13 → 6 (6.5에서 버림)

③ Flatten

전부 곱하기: (5, 5, 32) → 5 × 5 × 32 = 800

④ RNN 출력
return_sequences=False(기본값): 마지막 시점만 → (None, units)
return_sequences=True: 모든 시점 → (None, 시점 수, units)
RNN을 두 층 쌓을 때는 앞 층이 True여야 해요(다음 RNN에 문장 전체를 넘겨야 하니까).
3부. 문제 푸는 순서 (매번 이렇게)
표를 그린다: 층 | 출력 shape | 파라미터
위에서부터 한 줄씩 채운다. 앞 층의 출력이 다음 층의 입력이에요.
Conv가 나오면 → same인지 valid인지, 입력 채널이 몇인지 먼저 확인한다.
RNN이 나오면 → 입력 차원(Embedding이 앞에 있으면 그 벡터 차원)과 units를 확인한다.
끝에서 파라미터를 다 더한다.
연습 하나 (같이 풀어 봐요)

(32, 32, 3) → Conv2D(32, 3) → MaxPool(2) → Flatten → Dense(10)

층	출력 shape	파라미터
Conv2D(32, 3)	32 − 3 + 1 = 30 → (30, 30, 32)	(3×3×3 + 1) × 32 = 896
MaxPool	30 ÷ 2 → (15, 15, 32)	0
Flatten	15×15×32 = 7,200	0
Dense(10)	10	(7,200 + 1) × 10 = 72,010

합계 72,906

정리: 파라미터는 "뉴런 하나가 받는 입력 수 + 1" × 뉴런 수만 기억하고, 층마다 입력 수가 무엇인지 바꿔 끼우면 돼요. 크기는 표를 그려 한 층씩 따라가면 돼요.

원하시면 이 개념으로 풀 수 있는 연습 문제를 보기 없이 몇 개 내 드릴게요.

11번 문제는 시계열에서 빈칸(결측치)을 어떻게 채우는지 묻는 문제예요.

데이터: [10, NaN, NaN, 16]
하루 1번 온도를 쟀는데, 2일째와 3일째에 기계가 고장 나서 기록이 빈 상황이라고 생각해 보세요.

날짜	1일	2일	3일	4일
값	10	?	?	16

이 빈칸을 채우는 방법이 여러 가지예요.

① LOCF: 직전 값으로 채우기 → [10, 10, 10, 16]

Last Observation Carried Forward, "마지막으로 본 값을 앞으로 끌고 간다"는 뜻이에요.

빈칸을 만나면 바로 앞에 있던 값을 복사해요.
2일째: 앞에 10 → 10
3일째: 앞이 (방금 채운) 10 → 10

비유: 일기를 이틀 못 썼을 때 "어제랑 똑같았겠지" 하고 어제 내용을 그대로 베끼는 거예요.

pandas에서는 ffill()(forward fill)이에요.

② NOCB: 직후 값으로 채우기 → [10, 16, 16, 16]

Next Observation Carried Backward, "다음에 본 값을 뒤로 끌고 온다"는 뜻이에요.

빈칸을 만나면 바로 뒤에 있는 값을 복사해요.
3일째: 뒤에 16 → 16
2일째: 뒤가 (방금 채운) 16 → 16

pandas에서는 bfill()(backward fill)이에요.

외우는 법: Last = 앞에 있던 거, Next = 뒤에 오는 거.

③ 선형 보간법: 직선으로 이어서 채우기 → [10, 12, 14, 16] ✅ 정답

10과 16 사이를 자로 직선을 그어서 그 선 위의 값을 읽는 거예요.

계산 방법

전체 차이: 16 − 10 = 6
간격이 몇 개인지 세기: 1일→2일, 2일→3일, 3일→4일 = 3칸
한 칸당 증가량: 6 ÷ 3 = 2
10에서 2씩 더하기: 10 → 12 → 14 → 16

자주 하는 실수: 빈칸이 2개라서 ÷ 2를 하는 거예요.
나누는 숫자는 빈칸 수가 아니라 간격 수예요. 점이 4개면 간격은 3개예요(나무 4그루 사이에 틈이 3개인 것처럼).

간격 수 = 빈칸 수 + 1

비유: 계단 높이가 10cm에서 16cm로 3계단 만에 올라가면, 한 계단에 2cm씩 똑같이 올라가는 거예요.

pandas에서는 interpolate()(기본값이 선형)예요.

④ 보기 c [10, 13, 13, 16]은 뭐였을까요?

양 끝의 평균((10 + 16) ÷ 2 = 13)으로 빈칸을 똑같이 채운 거예요. 평균값으로 채우는 방법도 실제로 있지만, 선형 보간은 아니에요. 값이 10에서 16으로 변해 가는 흐름을 무시하니까요.

한눈에 비교
방법	결과	특징
LOCF (ffill)	[10, 10, 10, 16]	직전 값 복사, 4일째에 값이 갑자기 뜀
NOCB (bfill)	[10, 16, 16, 16]	직후 값 복사, 2일째에 값이 갑자기 뜀
선형 보간 (interpolate)	[10, 12, 14, 16]	직선으로 부드럽게 이어짐
평균 대치	[10, 13, 13, 16]	흐름 무시

시계열에서 왜 중요할까요?
시계열은 순서가 있는 데이터라 앞뒤 값과 이어져 있어요. 그래서 그냥 전체 평균으로 채우기보다 앞뒤 값을 이용하는 LOCF·NOCB·보간법을 주로 써요.

하나 주의할 점이 있어요. NOCB와 보간법은 미래 값(16)을 보고 과거를 채우는 거예요. 예측 모델을 만들 때 테스트 구간의 값으로 훈련 구간을 채우면 데이터 누수가 생길 수 있어요. 실제 예측 상황이라면 그 시점에 이미 알고 있던 값만 쓰는 LOCF가 더 안전해요. 시험에서는 정의만 물어볼 가능성이 높지만, 실습 때는 기억해 두세요.